# Tavily Hybrid RAG with OracleDB

This notebook is the presentation version of `examples/demo_integration_cache_hybrid.py`.

It uses real services:

- Tavily API for fresh web results
- OracleDB as the local vector cache and memory store
- Cohere embeddings/reranking through the SDK default hybrid RAG path

The two demo moments to show are:

- `freshness_cache`: first run goes to Tavily, second run returns from Oracle cache
- `cache_then_memory`: first run goes to Tavily, second run falls through cache and returns from Oracle memory

## 1. Environment

The notebook reads credentials from the repository root `.env` file. Expected keys:

```bash
TAVILY_API_KEY=tvly-...
CO_API_KEY=...
ORACLE_USER=...
ORACLE_PASSWORD=...
ORACLE_DSN=host:1521/service
ORACLE_TABLE=TAVILY_DOCUMENTS
ORACLE_CACHE_TIMESTAMP_FIELD=ADDED_AT
```

Run the notebook with the project virtual environment where `python -m pip install -e ".[oracle]"` has already been installed.

In [1]:
from __future__ import annotations

import os
import re
import sys
from pathlib import Path
from typing import Any


def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "setup.py").exists() and (path / "tavily").exists():
            return path
    raise RuntimeError("Could not find tavily-python repo root.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def load_root_env_file(repo_root: Path) -> None:
    env_path = repo_root / ".env"
    if not env_path.exists():
        raise RuntimeError(f"Missing .env file at {env_path}")

    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith("export "):
            line = line[len("export "):].strip()
        if "=" not in line:
            continue

        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip()
        if (value.startswith("'") and value.endswith("'")) or (
            value.startswith('"') and value.endswith('"')
        ):
            value = value[1:-1]
        os.environ.setdefault(key, value)


load_root_env_file(REPO_ROOT)
print(f"Repo root: {REPO_ROOT}")
print("Loaded .env keys for Tavily/Oracle demo.")

Repo root: /Users/abhirajdustakar/Desktop/pod-4/gitlab/tavily-python
Loaded .env keys for Tavily/Oracle demo.


In [2]:
required_env = [
    "TAVILY_API_KEY",
    "CO_API_KEY",
    "ORACLE_USER",
    "ORACLE_PASSWORD",
    "ORACLE_DSN",
]
missing = [key for key in required_env if not os.getenv(key)]
if missing:
    raise RuntimeError(f"Missing required env vars: {', '.join(missing)}")

ORACLE_TABLE = os.getenv("ORACLE_TABLE", "TAVILY_DOCUMENTS").upper()
ORACLE_CACHE_TIMESTAMP_FIELD = os.getenv("ORACLE_CACHE_TIMESTAMP_FIELD", "ADDED_AT").upper()

print(f"Oracle table: {ORACLE_TABLE}")
print(f"Cache timestamp field: {ORACLE_CACHE_TIMESTAMP_FIELD}")

Oracle table: TAVILY_DOCUMENTS
Cache timestamp field: ADDED_AT


## 2. Connect to Oracle and prepare the demo table

The table is created if missing. If it already exists, the notebook adds any missing cache/memory metadata columns.

The embedding column uses `VECTOR(*, FLOAT32)` so it can accept the SDK's 1024-dimensional Cohere embeddings.

In [3]:
import oracledb
from tavily import TavilyHybridClient

IDENTIFIER = re.compile(r"^[A-Z_][A-Z0-9_]*$")


def validate_identifier(value: str, name: str) -> str:
    value = value.upper()
    if not IDENTIFIER.match(value):
        raise ValueError(f"Invalid Oracle identifier for {name}: {value}")
    return value


ORACLE_TABLE = validate_identifier(ORACLE_TABLE, "ORACLE_TABLE")
ORACLE_CACHE_TIMESTAMP_FIELD = validate_identifier(
    ORACLE_CACHE_TIMESTAMP_FIELD,
    "ORACLE_CACHE_TIMESTAMP_FIELD",
)

connection = oracledb.connect(
    user=os.environ["ORACLE_USER"],
    password=os.environ["ORACLE_PASSWORD"],
    dsn=os.environ["ORACLE_DSN"],
)

with connection.cursor() as cursor:
    cursor.execute("SELECT USER FROM dual")
    connected_user = cursor.fetchone()[0]

print(f"Connected as: {connected_user}")

Connected as: TAVILY_ITEST


In [4]:
def table_exists(table_name: str) -> bool:
    with connection.cursor() as cursor:
        cursor.execute(
            "SELECT COUNT(*) FROM USER_TABLES WHERE TABLE_NAME = :table_name",
            table_name=table_name,
        )
        return cursor.fetchone()[0] > 0


def existing_columns(table_name: str) -> set[str]:
    with connection.cursor() as cursor:
        cursor.execute(
            "SELECT COLUMN_NAME FROM USER_TAB_COLUMNS WHERE TABLE_NAME = :table_name",
            table_name=table_name,
        )
        return {row[0] for row in cursor.fetchall()}


def ensure_demo_table(table_name: str) -> None:
    if not table_exists(table_name):
        with connection.cursor() as cursor:
            cursor.execute(f"""
                CREATE TABLE {table_name} (
                    ID NUMBER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
                    CONTENT CLOB NOT NULL,
                    EMBEDDINGS VECTOR(*, FLOAT32) NOT NULL,
                    {ORACLE_CACHE_TIMESTAMP_FIELD} TIMESTAMP DEFAULT SYSTIMESTAMP,
                    MEMORY_SCOPE VARCHAR2(30),
                    EXPIRES_AT TIMESTAMP WITH TIME ZONE,
                    LAST_SEEN_AT TIMESTAMP WITH TIME ZONE,
                    QUERY_COUNT NUMBER
                )
            """)
        connection.commit()
        print(f"Created table: {table_name}")
        return

    columns = existing_columns(table_name)
    required_columns = {
        ORACLE_CACHE_TIMESTAMP_FIELD: "TIMESTAMP DEFAULT SYSTIMESTAMP",
        "MEMORY_SCOPE": "VARCHAR2(30)",
        "EXPIRES_AT": "TIMESTAMP WITH TIME ZONE",
        "LAST_SEEN_AT": "TIMESTAMP WITH TIME ZONE",
        "QUERY_COUNT": "NUMBER",
    }
    with connection.cursor() as cursor:
        for column_name, column_type in required_columns.items():
            if column_name in columns:
                print(f"Column exists: {column_name}")
                continue
            cursor.execute(f"ALTER TABLE {table_name} ADD ({column_name} {column_type})")
            print(f"Added column: {column_name}")
    connection.commit()


ensure_demo_table(ORACLE_TABLE)

Column exists: ADDED_AT
Column exists: MEMORY_SCOPE
Column exists: EXPIRES_AT
Column exists: LAST_SEEN_AT
Column exists: QUERY_COUNT


In [5]:
with connection.cursor() as cursor:
    cursor.execute(
        """
        SELECT COLUMN_NAME, DATA_TYPE
        FROM USER_TAB_COLUMNS
        WHERE TABLE_NAME = :table_name
        ORDER BY COLUMN_ID
        """,
        table_name=ORACLE_TABLE,
    )
    rows = cursor.fetchall()

print(f"Columns in {ORACLE_TABLE}:")
for column_name, data_type in rows:
    print(f"  {column_name:<24} {data_type}")

Columns in TAVILY_DOCUMENTS:
  ID                       NUMBER
  CONTENT                  CLOB
  EMBEDDINGS               VECTOR
  ADDED_AT                 TIMESTAMP(6)
  MEMORY_SCOPE             VARCHAR2
  EXPIRES_AT               TIMESTAMP(6) WITH TIME ZONE
  LAST_SEEN_AT             TIMESTAMP(6) WITH TIME ZONE
  QUERY_COUNT              NUMBER


## 3. Shared demo helpers

`origin=foreign` means the result came from Tavily. `origin=local` means the result came from OracleDB.

In [7]:
def reset_demo_table() -> None:
    with connection.cursor() as cursor:
        cursor.execute(f"TRUNCATE TABLE {ORACLE_TABLE}")
    print(f"Reset table: {ORACLE_TABLE}")


def print_results(label: str, results: list[dict[str, Any]]) -> None:
    print(f"\n{label}")
    if not results:
        print("  (no results)")
        return

    for i, row in enumerate(results, start=1):
        content = str(row.get("content", ""))
        snippet = content[:90] + ("..." if len(content) > 90 else "")
        score = float(row.get("score", 0))
        origin = row.get("origin", "unknown")
        print(f"  {i}. origin={origin:<7} score={score:.4f} content={snippet}")

    local_count = sum(1 for row in results if row.get("origin") == "local")
    foreign_count = sum(1 for row in results if row.get("origin") == "foreign")
    print(f"  Summary: local={local_count}, foreign={foreign_count}, total={len(results)}")


def make_oracle_client(
    retrieval_mode: str,
    cache_score_threshold: float,
    memory_score_threshold: float = 0.6,
    persistence_depth: str = "cache_only",
    enable_oracle_memory_metadata: bool = False,
) -> TavilyHybridClient:
    return TavilyHybridClient(
        api_key=os.environ["TAVILY_API_KEY"],
        db_provider="oracle",
        connection=connection,
        table_name=ORACLE_TABLE,
        retrieval_mode=retrieval_mode,
        cache_timestamp_field=ORACLE_CACHE_TIMESTAMP_FIELD,
        cache_ttl_seconds=3600,
        cache_score_threshold=cache_score_threshold,
        memory_score_threshold=memory_score_threshold,
        memory_max_results=5,
        persistence_depth=persistence_depth,
        enable_oracle_memory_metadata=enable_oracle_memory_metadata,
    )


def run_search_twice(client: TavilyHybridClient, query: str) -> None:
    for run_number in range(1, 3):
        results = client.search(
            query=query,
            max_results=2,
            max_local=3,
            max_foreign=3,
            save_foreign=True,
        )
        print_results(f"Run #{run_number}", results)

## 4. Freshness cache demo

Expected presenter story:

- Run #1 misses Oracle cache, calls Tavily, and persists Tavily results to Oracle
- Run #2 finds fresh Oracle rows inside the TTL window and skips Tavily

In [8]:
reset_demo_table()

freshness_query = "Lionel Messi Newell's Old Boys childhood club history"
freshness_client = make_oracle_client(
    retrieval_mode="freshness_cache",
    cache_score_threshold=0.65,
)

run_search_twice(freshness_client, freshness_query)

Reset table: TAVILY_DOCUMENTS

Run #1
  1. origin=foreign score=0.7441 content=29 years ago today, Argentine club Newell's Old Boys registered a 6-year old boy named Lio...
  2. origin=foreign score=0.6927 content=By continuing, you agree to Instagram's Terms of Use and Privacy Policy. Lionel Messi's bo...
  Summary: local=0, foreign=2, total=2

Run #2
  1. origin=local   score=0.6842 content=29 years ago today, Argentine club Newell's Old Boys registered a 6-year old boy named Lio...
  2. origin=local   score=0.6578 content=By continuing, you agree to Instagram's Terms of Use and Privacy Policy. Lionel Messi's bo...
  Summary: local=2, foreign=0, total=2


## 5. Cache then memory demo

This mode checks three layers in order:

1. Fresh Oracle cache
2. Durable Oracle memory
3. Tavily fallback

The demo sets a high cache threshold so the second run falls through the cache tier and visibly returns from the memory tier.

In [9]:
reset_demo_table()

memory_query = "Oracle Database vector search cache then memory integration demo"
memory_client = make_oracle_client(
    retrieval_mode="cache_then_memory",
    cache_score_threshold=0.95,
    memory_score_threshold=0.6,
    persistence_depth="cache_plus_memory",
    enable_oracle_memory_metadata=True,
)

run_search_twice(memory_client, memory_query)

Reset table: TAVILY_DOCUMENTS

Run #1
  1. origin=foreign score=0.6027 content=# Oracle AI Vector Search : Demonstration. Oracle AI Vector Search is designed for **Artif...
  2. origin=foreign score=0.5522 content=# Oracle AI vector search - integration. Integrate with the Oracle AI vector search - vect...
  Summary: local=0, foreign=2, total=2

Run #2
  1. origin=local   score=0.6206 content=# Oracle AI vector search - integration. Integrate with the Oracle AI vector search - vect...
  Summary: local=1, foreign=0, total=1


## 6. Inspect persisted rows

This is a useful closing slide: it proves Tavily results have become Oracle retrieval context.

In [10]:
with connection.cursor() as cursor:
    cursor.execute(f"""
        SELECT NVL(MEMORY_SCOPE, '(none)') AS MEMORY_SCOPE,
               COUNT(*) AS ROW_COUNT
        FROM {ORACLE_TABLE}
        GROUP BY NVL(MEMORY_SCOPE, '(none)')
        ORDER BY MEMORY_SCOPE
    """)
    rows = cursor.fetchall()

print("Persisted rows by memory scope:")
for memory_scope, row_count in rows:
    print(f"  {memory_scope:<18} {row_count}")

Persisted rows by memory scope:
  cache_plus_memory  3


In [11]:
connection.close()
print("Oracle connection closed.")

Oracle connection closed.
